# Graph Structural Eval (All Conditions, Parallel)

Evaluates all model comparison CSVs against GT0513 across 4 conditions (A/B/C/D).
Uses 150s timeout per order, best-of-5 order strategies.

In [ ]:
import json
import random
import re
import signal
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import networkx as nx
import pandas as pd


In [ ]:
# ---- Config ----
ROOT = Path('..').resolve() if Path('..').joinpath('data').exists() else Path('.').resolve()

# Gold graph annotation (210 abstracts)
GT_JSON = ROOT / 'data' / 'raw' / 'EmpiriGraph-Psy_gold_annotation.json'
OUTPUT_DIR = ROOT / 'data' / 'output' / 'graph_eval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# format: (label, mode, csv_path)
# Run one at a time by uncommenting the desired spec
EVAL_SPECS = [
    # Best model: GPT-5.4 (Step 1, 5) + GPT-5.2 (Steps 2-4), 210 abstracts
    ('gpt54_gpt52_210', 'step5_1', ROOT / 'data' / 'output' / 'pipeline_gpt54_gpt52_210' / 'results_step5_1.csv'),

    # Single-model pipeline runs (165 abstracts)
    # ('gpt54_165',          'step5_1', ROOT / 'data' / 'output' / 'pipeline_gpt54_165' / 'results_step5_1.csv'),
    # ('gpt52_165',          'step5_1', ROOT / 'data' / 'output' / 'pipeline_gpt52_165' / 'results_step5_1.csv'),
    # ('gpt4o_165',          'step5_1', ROOT / 'data' / 'output' / 'pipeline_gpt4o_165' / 'results_step5_1.csv'),
    # ('claude_sonnet46_165','step5_1', ROOT / 'data' / 'output' / 'pipeline_claude_sonnet46_165' / 'results_step5_1.csv'),
    # ('claude_opus47_165',  'step5_1', ROOT / 'data' / 'output' / 'pipeline_claude_opus47_165' / 'results_step5_1.csv'),
    # ('gemini3flash_165',    'step5_1', ROOT / 'data' / 'output' / 'pipeline_gemini3flash_165' / 'results_step5_1.csv'),
    # ('deepseek_v4pro_210', 'step5_1', ROOT / 'data' / 'output' / 'pipeline_deepseek_v4pro_210' / 'results_step5_1.csv'),

    # Baselines
    # ('single_step_gpt54_210', 'step5_1', ROOT / 'data' / 'output' / 'single_step_gpt54_210_results.csv'),
]

TIMEOUT_PER_ORDER = 150.0
MAX_WORKERS = 12
ORDERS = ['degree_desc', 'degree_asc', 'label', 'random_11', 'random_29']
BEST_OF_ORDERS = True

print('GT_JSON:', GT_JSON)
print('OUTPUT_DIR:', OUTPUT_DIR)
for spec in EVAL_SPECS:
    print('EVAL:', spec)


In [ ]:
# Import all evaluation functions from external module
# This allows ProcessPoolExecutor to properly pickle the functions
from graph_eval_functions import (
    parse_gt_graph,
    parse_pred_graph,
    preprocess_graph_for_eval,
    to_higher_level_graph,
    to_type_agnostic_graph,
    evaluate_article,
)

In [ ]:
ETYPES = ['hierarchy', 'directional', 'correlational', 'moderation']

# 4 evaluation conditions
# (condition_key, gt_transform, pred_transform)
COND_TRANSFORMS = [
    ('A_typed',           lambda g: g,                                                    lambda g: g),
    ('B_higher_typed',    to_higher_level_graph,                                          to_higher_level_graph),
    ('C_agnostic',        to_type_agnostic_graph,                                         to_type_agnostic_graph),
    ('D_higher_agnostic', lambda g: to_type_agnostic_graph(to_higher_level_graph(g)),     lambda g: to_type_agnostic_graph(to_higher_level_graph(g))),
]

with open(GT_JSON, 'r', encoding='utf-8') as f:
    gt_articles = json.load(f)
gt_by_id = {int(a['id']): a for a in gt_articles}

all_summaries = []
all_details = {}

for label, mode, csv_path in EVAL_SPECS:
    df = pd.read_csv(csv_path)
    df['num_id'] = [int(str(x).replace('json_', '')) for x in df['article_id']]
    overlap = sorted(set(df['num_id']) & set(gt_by_id.keys()))
    print(f'\n=== {label} | overlap={len(overlap)} articles ===')

    # Build base preprocessed graphs once per spec
    gt_base  = {aid: preprocess_graph_for_eval(parse_gt_graph(gt_by_id[aid])) for aid in overlap}
    pred_base = {}
    for aid in overlap:
        row = df[df['num_id'] == aid].iloc[0]
        pred_base[aid] = preprocess_graph_for_eval(parse_pred_graph(row, mode))

    cond_details = {}
    for cond_key, gt_fn, pred_fn in COND_TRANSFORMS:
        jobs = [(aid, gt_fn(gt_base[aid]), pred_fn(pred_base[aid]),
                 TIMEOUT_PER_ORDER, ORDERS, BEST_OF_ORDERS) for aid in overlap]

        rows = []
        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = [ex.submit(evaluate_article, j) for j in jobs]
            done = 0
            for fut in as_completed(futs):
                rows.append(fut.result())
                done += 1
                if done % 10 == 0 or done == len(futs):
                    print(f'[{label}|{cond_key}] {done}/{len(futs)}')

        detail_df = pd.DataFrame(rows).sort_values('article_id')
        detail_csv = OUTPUT_DIR / f'graph_eval_{label}_{cond_key}_detail.csv'
        detail_df.to_csv(detail_csv, index=False)
        cond_details[cond_key] = detail_df

        sub = detail_df[detail_df['gt_n_edges'] > 0]
        sub_both = sub[sub['pred_n_edges'] > 0]
        s = {
            'label': label, 'mode': mode, 'condition': cond_key,
            'csv': str(csv_path),
            'articles_overlap': len(detail_df),
            'articles_gt_edges': len(sub),
            'articles_empty_pred': int((detail_df['pred_n_edges'] == 0).sum()),
            'mean_precision': float(sub['edge_p'].mean()) if len(sub) else 0.0,
            'mean_recall': float(sub['edge_r'].mean()) if len(sub) else 0.0,
            'mean_f1': float(sub['edge_f1'].mean()) if len(sub) else 0.0,
            'mean_f1_excl_empty': float(sub_both['edge_f1'].mean()) if len(sub_both) else 0.0,
            'timeout_per_order_sec': TIMEOUT_PER_ORDER,
            'order_strategies': '|'.join(ORDERS),
            'best_of_orders': bool(BEST_OF_ORDERS),
            'timeout_hit_articles': int(detail_df['any_timeout'].sum()),
            'detail_csv': str(detail_csv),
        }
        # per-type F1 (only articles with GT edges of that type) for typed conditions
        if cond_key in ('A_typed', 'B_higher_typed'):
            for et in ETYPES:
                has_gt = sub[sub[f'{et}_gt'] > 0]
                s[f'{et}_f1'] = float(has_gt[f'{et}_f1'].mean()) if len(has_gt) else float('nan')
                s[f'{et}_p']  = float(has_gt[f'{et}_p'].mean())  if len(has_gt) else float('nan')
                s[f'{et}_r']  = float(has_gt[f'{et}_r'].mean())  if len(has_gt) else float('nan')
                s[f'{et}_n']  = len(has_gt)
        all_summaries.append(s)

    # Merged per-article detail across all 4 conditions
    merged = cond_details['A_typed'][['article_id', 'gt_n_edges', 'pred_n_edges', 'any_timeout']].copy()
    for ck, cdf in cond_details.items():
        cols = {'edge_p': f'p_{ck}', 'edge_r': f'r_{ck}', 'edge_f1': f'f1_{ck}'}
        sub_merge = cdf[['article_id'] + list(cols.keys())].rename(columns=cols)
        if ck in ('A_typed', 'B_higher_typed'):
            for et in ETYPES:
                for m in ['_f1', '_p', '_r', '_gt', '_pred', '_matched']:
                    c = f'{et}{m}'
                    if c in cdf.columns:
                        sub_merge[f'{c}_{ck}'] = cdf[c].values
        merged = merged.merge(sub_merge, on='article_id', how='outer')
    merged_csv = OUTPUT_DIR / f'graph_eval_{label}_4conditions_detail.csv'
    merged.to_csv(merged_csv, index=False)
    print(f'Saved merged detail: {merged_csv}')

summary_df = pd.DataFrame(all_summaries)
summary_csv = OUTPUT_DIR / 'graph_eval_4conditions_parallel_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print('Saved summary:', summary_csv)

# Print results table
for s in all_summaries:
    print(f"\n[{s['label']} | {s['condition']}]  n={s['articles_gt_edges']}  empty={s['articles_empty_pred']}  timeouts={s['timeout_hit_articles']}")
    print(f"  Overall:  P={s['mean_precision']:.4f}  R={s['mean_recall']:.4f}  F1={s['mean_f1']:.4f}  (excl_empty F1={s['mean_f1_excl_empty']:.4f})")
    for et in ETYPES:
        if f'{et}_f1' in s:
            print(f"  {et:15s}: P={s[f'{et}_p']:.4f}  R={s[f'{et}_r']:.4f}  F1={s[f'{et}_f1']:.4f}  (n={s[f'{et}_n']})")

summary_df
